<a href="https://colab.research.google.com/github/be-ayush/ai-ml-learning/blob/main/HOML/HOML3_Chapter10_PytorchNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])

In [2]:
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [3]:
X.shape

torch.Size([2, 3])

In [4]:
X.dtype

torch.float32

In [5]:
X[0,1]

tensor(4.)

In [6]:
10 * (X + 1.0)

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [7]:
X.exp()

tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])

In [8]:
X.mean()

tensor(3.8333)

In [10]:
X.max(dim=1)

torch.return_types.max(
values=tensor([7., 6.]),
indices=tensor([2, 2]))

In [11]:
X @ X.T

tensor([[66., 56.],
        [56., 49.]])

In [12]:
import numpy as np

X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [13]:
torch.tensor(np.array([[1, 2], [3, 4]]))

tensor([[1, 2],
        [3, 4]])

In [20]:
X[:, 1] = -99
X

tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In [17]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])

In [21]:
X.relu_()

tensor([[1., 0., 7.],
        [2., 0., 6.]])

In [22]:
X

tensor([[1., 0., 7.],
        [2., 0., 6.]])

In [23]:
if torch.cuda.is_available():
  device = "cuda"
elif torch.backends.mps.is_available():
  device = "mps"
else:
  device = "cpu"

In [24]:
device

'cuda'

In [25]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

In [26]:
M = M.to(device)

In [27]:
M.device

device(type='cuda', index=0)

In [28]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

In [29]:
M = torch.rand((1000, 1000))
%timeit M @ M.T

5.74 ms ± 655 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [30]:
M = torch.rand((1000, 1000), device=device)
%timeit M @ M.T

611 µs ± 19.9 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [34]:
x = torch.tensor(5.0, requires_grad=True)
f = x**2
print(f)

f.backward()
x.grad

tensor(25., grad_fn=<PowBackward0>)


tensor(10.)

In [35]:
learning_rate = 0.1
with torch.no_grad():
  x -= learning_rate * x.grad

In [36]:
x

tensor(4., requires_grad=True)

In [37]:
x.grad.zero_()

tensor(0.)

In [38]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for interation in range(100):
  f = x**2 # forward pass
  f.backward() # backward pass compute the gradients
  with torch.no_grad():
    x -= learning_rate * x.grad # update
  x.grad.zero_() # reset gradients for the next iteration

In [39]:
x

tensor(1.0185e-09, requires_grad=True)

In [41]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split


In [42]:
housing = fetch_california_housing(as_frame=False)
X, y = housing.data, housing.target

# First split: 80% for training, 20% for temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Second split: Half of the temp set for validation, half for test (10% each of original)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Validation set shape: {X_val.shape}, {y_val.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (16512, 8), (16512,)
Validation set shape: (2064, 8), (2064,)
Test set shape: (2064, 8), (2064,)


In [43]:
X_train = torch.FloatTensor(X_train)
X_val = torch.FloatTensor(X_val)
X_test = torch.FloatTensor(X_test)

means = X_train.mean(dim=0)
stds = X_train.std(dim=0)

X_train = (X_train - means) / stds
X_val = (X_val - means) / stds
X_test = (X_test - means) / stds

In [44]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_val = torch.FloatTensor(y_val).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [45]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [46]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
  y_pred = X_train @ w + b
  loss = ((y_pred - y_train) ** 2).mean()
  loss.backward()
  with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad
  w.grad.zero_()
  b.grad.zero_()
  print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

Epoch 1, Loss: 16.044116973876953
Epoch 2, Loss: 4.6996660232543945
Epoch 3, Loss: 2.1376543045043945
Epoch 4, Loss: 1.2627449035644531
Epoch 5, Loss: 0.9295794367790222
Epoch 6, Loss: 0.7914232611656189
Epoch 7, Loss: 0.7266561388969421
Epoch 8, Loss: 0.6907869577407837
Epoch 9, Loss: 0.6671210527420044
Epoch 10, Loss: 0.6492294669151306
Epoch 11, Loss: 0.6345414519309998
Epoch 12, Loss: 0.6219596862792969
Epoch 13, Loss: 0.610961377620697
Epoch 14, Loss: 0.6012550592422485
Epoch 15, Loss: 0.5926494002342224
Epoch 16, Loss: 0.5850006937980652
Epoch 17, Loss: 0.5781919956207275
Epoch 18, Loss: 0.5721242427825928
Epoch 19, Loss: 0.5667114853858948
Epoch 20, Loss: 0.5618786215782166


In [47]:
X_new = X_test[:3]
with torch.no_grad(): # during inference no_grad helps speeding up the inference as torch does not need to keep track of the computation graph
  y_pred = X_new @ w + b
y_pred

tensor([[1.9009],
        [2.1525],
        [1.9111]])

In [48]:
import torch.nn as nn

torch.manual_seed(42)
n_features = X_train.shape[1]
model = nn.Linear(in_features = n_features, out_features = 1)

In [49]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [50]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [51]:
model(X_train[:2])

tensor([[0.7173],
        [1.0659]], grad_fn=<AddmmBackward0>)

In [52]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [53]:
def train_batch_gd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

In [54]:
train_batch_gd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1, Loss: 4.2877516746521
Epoch 2, Loss: 0.7712735533714294
Epoch 3, Loss: 0.6180822253227234
Epoch 4, Loss: 0.5989888310432434
Epoch 5, Loss: 0.5885871052742004
Epoch 6, Loss: 0.5802777409553528
Epoch 7, Loss: 0.5731872916221619
Epoch 8, Loss: 0.5670098662376404
Epoch 9, Loss: 0.5615824460983276
Epoch 10, Loss: 0.5567958950996399
Epoch 11, Loss: 0.5525659918785095
Epoch 12, Loss: 0.5488236546516418
Epoch 13, Loss: 0.5455095171928406
Epoch 14, Loss: 0.5425723791122437
Epoch 15, Loss: 0.5399671196937561
Epoch 16, Loss: 0.5376547574996948
Epoch 17, Loss: 0.5356006622314453
Epoch 18, Loss: 0.5337744951248169
Epoch 19, Loss: 0.5321498513221741
Epoch 20, Loss: 0.5307032465934753


In [55]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = model(X_new)
y_pred

tensor([[1.9227],
        [2.3950],
        [1.9292]])

In [58]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [60]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_batch_gd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1, Loss: 1.050593376159668
Epoch 2, Loss: 0.8649062514305115
Epoch 3, Loss: 0.7824119329452515
Epoch 4, Loss: 0.7311302423477173
Epoch 5, Loss: 0.7012783885002136
Epoch 6, Loss: 0.680791974067688
Epoch 7, Loss: 0.6667258143424988
Epoch 8, Loss: 0.6556363105773926
Epoch 9, Loss: 0.6467151641845703
Epoch 10, Loss: 0.6387865543365479
Epoch 11, Loss: 0.6317043900489807
Epoch 12, Loss: 0.6250802874565125
Epoch 13, Loss: 0.6188583374023438
Epoch 14, Loss: 0.6129239201545715
Epoch 15, Loss: 0.6072747111320496
Epoch 16, Loss: 0.6018303632736206
Epoch 17, Loss: 0.5965894460678101
Epoch 18, Loss: 0.5915117859840393
Epoch 19, Loss: 0.5866151452064514
Epoch 20, Loss: 0.581882894039154


In [61]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [62]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)
model.to(device)

Sequential(
  (0): Linear(in_features=8, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=40, bias=True)
  (3): ReLU()
  (4): Linear(in_features=40, out_features=1, bias=True)
)

In [63]:
learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [64]:
def train(model, optimizer, criterion, train_loader, n_epochs):
  model.train()
  for epoch in range(n_epochs):
    total_loss = 0.
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, Loss: {mean_loss}")

In [65]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1, Loss: 0.5788575623819764
Epoch 2, Loss: 0.42620307803442775
Epoch 3, Loss: 0.39667273334465747
Epoch 4, Loss: 0.3811814212926136
Epoch 5, Loss: 0.37283170086064543
Epoch 6, Loss: 0.37113179270784524
Epoch 7, Loss: 0.35487497388565725
Epoch 8, Loss: 0.3468171807598005
Epoch 9, Loss: 0.33949555136090104
Epoch 10, Loss: 0.3370385218931492
Epoch 11, Loss: 0.33057447784226535
Epoch 12, Loss: 0.32476205017793086
Epoch 13, Loss: 0.3215446251971546
Epoch 14, Loss: 0.3163453028212453
Epoch 15, Loss: 0.31319151307607807
Epoch 16, Loss: 0.3112654638180668
Epoch 17, Loss: 0.3104076484117166
Epoch 18, Loss: 0.3079151000336621
Epoch 19, Loss: 0.30481745953881
Epoch 20, Loss: 0.30397057373712


In [66]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
  model.eval()
  metrics = []
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric = metric_fn(y_pred, y_batch)
      metrics.append(metric)
  return aggregate_fn(torch.stack(metrics))

In [67]:
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32)
val_mse = evaluate(model, val_loader, mse)
val_mse

tensor(0.3063, device='cuda:0')

In [68]:
def rmse(y_pred, y_true):
  return ((y_pred - y_true) ** 2).mean().sqrt()

evaluate(model, val_loader, rmse)

tensor(0.5376, device='cuda:0')

In [69]:
val_mse.sqrt()

tensor(0.5534, device='cuda:0')

In [70]:
evaluate(model, val_loader, mse, aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(0.5534, device='cuda:0')

In [71]:
%pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 63.8 MB/s eta 0:00:00


In [73]:
import torchmetrics

def evaluate_torchmetrics(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch)
  return metric.compute()

In [74]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_torchmetrics(model, val_loader, rmse)

tensor(0.5524, device='cuda:0')

In [77]:
def train_with_validation(model, optimizer, criterion, train_loader, val_loader, metric, n_epochs):
  model.train()
  train_losses = []
  val_rmses = []
  for epoch in range(n_epochs):
    total_loss = 0.
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    train_losses.append(mean_loss)
    print(f"Epoch {epoch + 1}, Train Loss: {mean_loss}")
    rmse = evaluate_torchmetrics(model, val_loader, metric)
    val_rmses.append(rmse.item())
    print(f"Epoch {epoch + 1}, Validation RMSE: {rmse}")
  return train_losses, val_rmses

In [85]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)
model.to(device)
learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_losses, val_rmses = train_with_validation(model, optimizer, rmse, train_loader, val_loader, rmse, n_epochs)

Epoch 1, Train Loss: 0.7943456085730892
Epoch 1, Validation RMSE: 0.7039139866828918
Epoch 2, Train Loss: 0.6538953622297723
Epoch 2, Validation RMSE: 0.6622180938720703
Epoch 3, Train Loss: 0.6260304329122683
Epoch 3, Validation RMSE: 0.6408485770225525
Epoch 4, Train Loss: 0.6129372210581173
Epoch 4, Validation RMSE: 0.6370483636856079
Epoch 5, Train Loss: 0.599423017035159
Epoch 5, Validation RMSE: 0.6215532422065735
Epoch 6, Train Loss: 0.5922574208572854
Epoch 6, Validation RMSE: 0.6121801733970642
Epoch 7, Train Loss: 0.5876135796077492
Epoch 7, Validation RMSE: 0.6088212728500366
Epoch 8, Train Loss: 0.5805333742453146
Epoch 8, Validation RMSE: 0.6030377745628357
Epoch 9, Train Loss: 0.5754605543243793
Epoch 9, Validation RMSE: 0.6058098077774048
Epoch 10, Train Loss: 0.569544523376827
Epoch 10, Validation RMSE: 0.5955896377563477
Epoch 11, Train Loss: 0.5653303967312325
Epoch 11, Validation RMSE: 0.6047683358192444
Epoch 12, Train Loss: 0.5638721420090328
Epoch 12, Validation R

In [86]:
import plotly.graph_objects as go

epochs = list(range(1, n_epochs + 1))

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_losses, mode='lines', name='Training Loss (MSE)'))
fig.add_trace(go.Scatter(x=epochs, y=val_rmses, mode='lines', name='Validation RMSE'))

fig.update_layout(
    title='Learning Curve',
    xaxis_title='Epoch',
    yaxis_title='Metric Value',
    hovermode='x unified'
)

fig.show()

In [4]:
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

if torch.cuda.is_available():
  device = "cuda"
elif torch.backends.mps.is_available():
  device = "mps"
else:
  device = "cpu"

housing = fetch_california_housing(as_frame=False)
X, y = housing.data, housing.target

# First split: 80% for training, 20% for temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Second split: Half of the temp set for validation, half for test (10% each of original)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Validation set shape: {X_val.shape}, {y_val.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

X_train = torch.FloatTensor(X_train)
X_val = torch.FloatTensor(X_val)
X_test = torch.FloatTensor(X_test)

means = X_train.mean(dim=0)
stds = X_train.std(dim=0)

X_train = (X_train - means) / stds
X_val = (X_val - means) / stds
X_test = (X_test - means) / stds

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_val = torch.FloatTensor(y_val).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32)

Training set shape: (16512, 8), (16512,)
Validation set shape: (2064, 8), (2064,)
Test set shape: (2064, 8), (2064,)


In [3]:
%pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 46.4 MB/s eta 0:00:00


In [11]:
import torchmetrics
import plotly.graph_objects as go

def evaluate_torchmetrics(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch)
  return metric.compute()


def train_with_validation(model, optimizer, criterion, train_loader, val_loader, metric, n_epochs):
  model.train()
  train_losses = []
  val_rmses = []
  for epoch in range(n_epochs):
    total_loss = 0.
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    train_losses.append(mean_loss)
    print(f"Epoch {epoch + 1}, Train RMSE: {mean_loss}")
    rmse = evaluate_torchmetrics(model, val_loader, metric)
    val_rmses.append(rmse.item())
    print(f"Epoch {epoch + 1}, Validation RMSE: {rmse}")
  return train_losses, val_rmses

def plot_loss_curves(train_losses, val_rmses, n_epochs):
  epochs = list(range(1, 20 + 1))

  fig = go.Figure()
  fig.add_trace(go.Scatter(x=epochs, y=train_losses, mode='lines', name='Training Loss (MSE)'))
  fig.add_trace(go.Scatter(x=epochs, y=val_rmses, mode='lines', name='Validation RMSE'))

  fig.update_layout(
      title='Learning Curve',
      xaxis_title='Epoch',
      yaxis_title='Metric Value',
      hovermode='x unified'
  )

  fig.show()

In [8]:
import torch.nn as nn

class WideAndDeep(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features, 50),
        nn.ReLU(),
        nn.Linear(50, 40),
        nn.ReLU()
    )
    self.output_layer = nn.Linear(40 + n_features, 1)

  def forward(self, X):
    deep_out = self.deep_stack(X)
    wide_and_deep = torch.concat([X, deep_out], dim = 1)
    return self.output_layer(wide_and_deep)

n_features = X_train.shape[1]

torch.manual_seed(42)
model = WideAndDeep(n_features).to(device)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)

train_losses, val_rmses = train_with_validation(model, optimizer, rmse, train_loader, val_loader, rmse, n_epochs=20)

Epoch 1, Train RMSE: 1.5235185884228049
Epoch 1, Validation RMSE: 0.8752408623695374
Epoch 2, Train RMSE: 0.8125779095084168
Epoch 2, Validation RMSE: 0.8103594183921814
Epoch 3, Train RMSE: 0.7695326313029888
Epoch 3, Validation RMSE: 0.7853966355323792
Epoch 4, Train RMSE: 0.7463994944511458
Epoch 4, Validation RMSE: 0.7659451961517334
Epoch 5, Train RMSE: 0.7308895955252093
Epoch 5, Validation RMSE: 0.7527576684951782
Epoch 6, Train RMSE: 0.7147097187333329
Epoch 6, Validation RMSE: 0.7393710017204285
Epoch 7, Train RMSE: 0.7043788629446843
Epoch 7, Validation RMSE: 0.7302111983299255
Epoch 8, Train RMSE: 0.6943003580898277
Epoch 8, Validation RMSE: 0.7216317653656006
Epoch 9, Train RMSE: 0.6883178267252538
Epoch 9, Validation RMSE: 0.7157433032989502
Epoch 10, Train RMSE: 0.6811315724553988
Epoch 10, Validation RMSE: 0.7091732025146484
Epoch 11, Train RMSE: 0.6766205449090448
Epoch 11, Validation RMSE: 0.7055613994598389
Epoch 12, Train RMSE: 0.6720558109209519
Epoch 12, Validation

In [10]:

epochs = list(range(1, 20 + 1))

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_losses, mode='lines', name='Training Loss (MSE)'))
fig.add_trace(go.Scatter(x=epochs, y=val_rmses, mode='lines', name='Validation RMSE'))

fig.update_layout(
    title='Learning Curve',
    xaxis_title='Epoch',
    yaxis_title='Metric Value',
    hovermode='x unified'
)

fig.show()

In [13]:
import torch.nn as nn

class WideAndDeepV2(nn.Module):
  def __init__(self, n_total_features):
    super().__init__()
    # Input to deep_stack is X_deep, which is X[:, 2:]. So its input features are n_total_features - 2
    self.deep_stack = nn.Sequential(
        nn.Linear(n_total_features - 2, 50),
        nn.ReLU(),
        nn.Linear(50, 40),
        nn.ReLU()
    )
    # Input to output_layer is torch.concat([X_wide, deep_output]).
    # X_wide has 5 features (X[:, :5])
    # deep_output has 40 features (output of self.deep_stack)
    # Total input features for output_layer = 5 + 40 = 45
    self.output_layer = nn.Linear(40 + 5, 1)

  def forward(self, X_wide, X_deep):
    deep_out = self.deep_stack(X_deep)
    wide_and_deep = torch.concat([X_wide, deep_out], dim=1)
    return self.output_layer(wide_and_deep)

# Example usage with n_features = X_train.shape[1]
n_total_features = X_train.shape[1]
torch.manual_seed(42)
model_v2_updated = WideAndDeepV2(n_total_features).to(device)

print(model_v2_updated)

train_data_wide_deep_2 = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wide_deep_2 = DataLoader(train_data_wide_deep_2, batch_size=32, shuffle=True)

val_data_wide_deep_2 = TensorDataset(X_val[:, :5], X_val[:, 2:], y_val)
val_loader_wide_deep_2 = DataLoader(val_data_wide_deep_2, batch_size=32, shuffle=True)

WideAndDeepV2(
  (deep_stack): Sequential(
    (0): Linear(in_features=6, out_features=50, bias=True)
    (1): ReLU()
    (2): Linear(in_features=50, out_features=40, bias=True)
    (3): ReLU()
  )
  (output_layer): Linear(in_features=45, out_features=1, bias=True)
)


In [15]:
import torchmetrics
import plotly.graph_objects as go

def evaluate_torchmetrics(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch_wide, X_batch_deep, y_batch in data_loader:
      X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to(device), X_batch_deep.to(device), y_batch.to(device)
      y_pred = model(X_batch_wide, X_batch_deep)
      metric.update(y_pred, y_batch)
  return metric.compute()


def train_with_validation(model, optimizer, criterion, train_loader, val_loader, metric, n_epochs):
  model.train()
  train_losses = []
  val_rmses = []
  for epoch in range(n_epochs):
    total_loss = 0.
    for X_batch_wide, X_batch_deep, y_batch in train_loader:
      X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to(device), X_batch_deep.to(device), y_batch.to(device)
      y_pred = model(X_batch_wide, X_batch_deep)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    train_losses.append(mean_loss)
    print(f"Epoch {epoch + 1}, Train RMSE: {mean_loss}")
    rmse = evaluate_torchmetrics(model, val_loader, metric)
    val_rmses.append(rmse.item())
    print(f"Epoch {epoch + 1}, Validation RMSE: {rmse}")
  return train_losses, val_rmses

learning_rate = 0.002
optimizer = torch.optim.SGD(model_v2_updated.parameters(), lr=learning_rate)
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)

train_losses, val_rmses = train_with_validation(model_v2_updated, optimizer, rmse, train_loader_wide_deep_2, val_loader_wide_deep_2, rmse, n_epochs=20)

plot_loss_curves(train_losses, val_rmses, n_epochs=20)

Epoch 1, Train RMSE: 1.538941668563111
Epoch 1, Validation RMSE: 0.9422245025634766
Epoch 2, Train RMSE: 0.8455901765084082
Epoch 2, Validation RMSE: 0.8073486089706421
Epoch 3, Train RMSE: 0.7735827542668166
Epoch 3, Validation RMSE: 0.7768973112106323
Epoch 4, Train RMSE: 0.7427609015920366
Epoch 4, Validation RMSE: 0.7570227384567261
Epoch 5, Train RMSE: 0.7242360302182131
Epoch 5, Validation RMSE: 0.7425403594970703
Epoch 6, Train RMSE: 0.7101892507237981
Epoch 6, Validation RMSE: 0.7323050498962402
Epoch 7, Train RMSE: 0.7001768071637597
Epoch 7, Validation RMSE: 0.7245019674301147
Epoch 8, Train RMSE: 0.6940362835808318
Epoch 8, Validation RMSE: 0.717898964881897
Epoch 9, Train RMSE: 0.6878874414535456
Epoch 9, Validation RMSE: 0.7134981751441956
Epoch 10, Train RMSE: 0.6836731266952301
Epoch 10, Validation RMSE: 0.7094611525535583
Epoch 11, Train RMSE: 0.6801860266646673
Epoch 11, Validation RMSE: 0.7058236002922058
Epoch 12, Train RMSE: 0.6762706675501757
Epoch 12, Validation R

In [16]:
import torch.nn as nn

class WideAndDeepV3(nn.Module):
  def __init__(self, n_total_features):
    super().__init__()
    # Input to deep_stack is X_deep, which is X[:, 2:]. So its input features are n_total_features - 2
    self.deep_stack = nn.Sequential(
        nn.Linear(n_total_features - 2, 50),
        nn.ReLU(),
        nn.Linear(50, 40),
        nn.ReLU()
    )
    # Input to output_layer is torch.concat([X_wide, deep_output]).
    # X_wide has 5 features (X[:, :5])
    # deep_output has 40 features (output of self.deep_stack)
    # Total input features for output_layer = 5 + 40 = 45
    self.output_layer = nn.Linear(40 + 5, 1)
    self.aux_output = nn.Linear(40, 1)

  def forward(self, X_wide, X_deep):
    deep_out = self.deep_stack(X_deep)
    wide_and_deep = torch.concat([X_wide, deep_out], dim=1)
    return self.output_layer(wide_and_deep), self.aux_output(deep_out)

# Example usage with n_features = X_train.shape[1]
n_total_features = X_train.shape[1]
torch.manual_seed(42)
model_v3_updated = WideAndDeepV3(n_total_features).to(device)

print(model_v3_updated)

WideAndDeepV3(
  (deep_stack): Sequential(
    (0): Linear(in_features=6, out_features=50, bias=True)
    (1): ReLU()
    (2): Linear(in_features=50, out_features=40, bias=True)
    (3): ReLU()
  )
  (output_layer): Linear(in_features=45, out_features=1, bias=True)
  (aux_output): Linear(in_features=40, out_features=1, bias=True)
)


In [17]:
import torchmetrics
import plotly.graph_objects as go

def evaluate_torchmetrics(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch_wide, X_batch_deep, y_batch in data_loader:
      X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to(device), X_batch_deep.to(device), y_batch.to(device)
      y_pred, y_pred_aux = model(X_batch_wide, X_batch_deep)
      # For evaluation, we typically care about the main output's performance
      metric.update(y_pred, y_batch)
  return metric.compute()


def train_with_validation(model, optimizer, criterion, train_loader, val_loader, metric, n_epochs):
  model.train()
  train_losses = []
  val_rmses = []
  for epoch in range(n_epochs):
    total_loss = 0.
    for X_batch_wide, X_batch_deep, y_batch in train_loader:
      X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to(device), X_batch_deep.to(device), y_batch.to(device)
      y_pred, y_pred_aux = model(X_batch_wide, X_batch_deep)
      main_loss = criterion(y_pred, y_batch)
      aux_loss = criterion(y_pred_aux, y_batch)
      loss = 0.8 * main_loss + 0.2 * aux_loss # Combined loss
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
    mean_loss = total_loss / len(train_loader)
    train_losses.append(mean_loss)
    print(f"Epoch {epoch + 1}, Train Loss (Combined): {mean_loss}")
    rmse = evaluate_torchmetrics(model, val_loader, metric)
    val_rmses.append(rmse.item())
    print(f"Epoch {epoch + 1}, Validation RMSE: {rmse}")
  return train_losses, val_rmses

learning_rate = 0.002
optimizer = torch.optim.SGD(model_v3_updated.parameters(), lr=learning_rate) # Use model_v3_updated
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)

train_losses, val_rmses = train_with_validation(model_v3_updated, optimizer, rmse, train_loader_wide_deep_2, val_loader_wide_deep_2, rmse, n_epochs=20)

plot_loss_curves(train_losses, val_rmses, n_epochs=20)

Epoch 1, Train Loss (Combined): 1.739317079962686
Epoch 1, Validation RMSE: 1.0379672050476074
Epoch 2, Train Loss (Combined): 1.0259376727333365
Epoch 2, Validation RMSE: 0.8349251747131348
Epoch 3, Train Loss (Combined): 0.8828733799069427
Epoch 3, Validation RMSE: 0.7969849705696106
Epoch 4, Train Loss (Combined): 0.8386744648911232
Epoch 4, Validation RMSE: 0.7769841551780701
Epoch 5, Train Loss (Combined): 0.8162940645864768
Epoch 5, Validation RMSE: 0.7610563635826111
Epoch 6, Train Loss (Combined): 0.7979708689936372
Epoch 6, Validation RMSE: 0.7484540939331055
Epoch 7, Train Loss (Combined): 0.7835136364365733
Epoch 7, Validation RMSE: 0.7374926805496216
Epoch 8, Train Loss (Combined): 0.7732505663767342
Epoch 8, Validation RMSE: 0.7298049926757812
Epoch 9, Train Loss (Combined): 0.7632548084208207
Epoch 9, Validation RMSE: 0.7231818437576294
Epoch 10, Train Loss (Combined): 0.755575455840706
Epoch 10, Validation RMSE: 0.7183182835578918
Epoch 11, Train Loss (Combined): 0.74722

In [18]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_val_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor
)

test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor
)

torch.manual_seed(42)

train_data, valid_data = torch.utils.data.random_split(
    train_and_val_data, [55000, 5000],
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 191kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.50MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.4MB/s]


In [19]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [24]:
train_and_val_data.classes[train_data[0][1]]

'Ankle boot'

In [27]:
class ImageClassifier(nn.Module):
  def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
    super().__init__()
    self.mlp = nn.Sequential(
        nn.Flatten(),
        nn.Linear(n_inputs, n_hidden1),
        nn.ReLU(),
        nn.Linear(n_hidden1, n_hidden2),
        nn.ReLU(),
        nn.Linear(n_hidden2, n_classes)
    )

  def forward(self, X):
    return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifier(n_inputs = 28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10)
model.to(device)
cross_entropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)



In [29]:
def evaluate_torchmetrics_image(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch)
  return metric.compute()

def train_one_epoch(model, optimizer, criterion, train_loader):
  model.train() # Set model to training mode for this epoch
  total_loss = 0.
  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    y_pred = model(X_batch)
    loss = criterion(y_pred, y_batch)
    total_loss += loss.item()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
  return total_loss / len(train_loader)

def run_training_and_evaluation(model, optimizer, criterion, train_loader, val_loader, metric, n_epochs):
  train_losses = []
  train_accuracies = []
  val_accuracies = []
  for epoch in range(n_epochs):
    # --- Training Loop for one epoch ---
    mean_train_loss = train_one_epoch(model, optimizer, criterion, train_loader)
    train_losses.append(mean_train_loss)
    print(f"Epoch {epoch + 1}, Train Loss: {mean_train_loss:.4f}")

    # --- Evaluation on Training Set ---
    train_acc = evaluate_torchmetrics_image(model, train_loader, metric)
    train_accuracies.append(train_acc.item())
    print(f"Epoch {epoch + 1}, Train Accuracy: {train_acc:.4f}")

    # --- Evaluation on Validation Set ---
    val_acc = evaluate_torchmetrics_image(model, val_loader, metric)
    val_accuracies.append(val_acc.item())
    print(f"Epoch {epoch + 1}, Validation Accuracy: {val_acc:.4f}")
  return train_losses, train_accuracies, val_accuracies

# Assuming n_epochs is already defined
n_epochs = 20

train_losses_img, train_accuracies_img, val_accuracies_img = run_training_and_evaluation(model, optimizer, cross_entropy, train_loader, val_loader, accuracy, n_epochs)

Epoch 1, Train Loss: 0.5020
Epoch 1, Train Accuracy: 0.8270
Epoch 1, Validation Accuracy: 0.8180
Epoch 2, Train Loss: 0.4911
Epoch 2, Train Accuracy: 0.8222
Epoch 2, Validation Accuracy: 0.8080
Epoch 3, Train Loss: 0.4822
Epoch 3, Train Accuracy: 0.8303
Epoch 3, Validation Accuracy: 0.8192
Epoch 4, Train Loss: 0.4740
Epoch 4, Train Accuracy: 0.8369
Epoch 4, Validation Accuracy: 0.8242
Epoch 5, Train Loss: 0.4676
Epoch 5, Train Accuracy: 0.8386
Epoch 5, Validation Accuracy: 0.8304
Epoch 6, Train Loss: 0.4618
Epoch 6, Train Accuracy: 0.8404
Epoch 6, Validation Accuracy: 0.8328
Epoch 7, Train Loss: 0.4563
Epoch 7, Train Accuracy: 0.8407
Epoch 7, Validation Accuracy: 0.8296
Epoch 8, Train Loss: 0.4515
Epoch 8, Train Accuracy: 0.8367
Epoch 8, Validation Accuracy: 0.8242
Epoch 9, Train Loss: 0.4465
Epoch 9, Train Accuracy: 0.8445
Epoch 9, Validation Accuracy: 0.8320
Epoch 10, Train Loss: 0.4424
Epoch 10, Train Accuracy: 0.8392
Epoch 10, Validation Accuracy: 0.8276
Epoch 11, Train Loss: 0.438

In [30]:
import plotly.graph_objects as go

epochs = list(range(1, n_epochs + 1))

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_losses_img, mode='lines', name='Training Loss (CrossEntropy)'))
fig.add_trace(go.Scatter(x=epochs, y=train_accuracies_img, mode='lines', name='Training Accuracy'))
fig.add_trace(go.Scatter(x=epochs, y=val_accuracies_img, mode='lines', name='Validation Accuracy'))

fig.update_layout(
    title='Learning Curve for Image Classifier',
    xaxis_title='Epoch',
    yaxis_title='Metric Value',
    hovermode='x unified'
)

fig.show()

In [31]:
model.eval()
X_new, y_new = next(iter(val_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
  y_pred_logits = model(X_new)

y_pred = y_pred_logits.argmax(dim=1)
print(y_pred)
print([train_and_val_data.classes[index] for index in y_pred])

tensor([7, 4, 2], device='cuda:0')
['Sneaker', 'Coat', 'Pullover']


In [33]:
import torch.nn.functional as F

y_proba = F.softmax(y_pred_logits, dim=1)


print(y_proba.round(decimals=3))

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0640, 0.0000, 0.8690, 0.0010,
         0.0650],
        [0.0000, 0.0000, 0.0320, 0.0000, 0.9670, 0.0000, 0.0010, 0.0000, 0.0000,
         0.0000],
        [0.0020, 0.0000, 0.9210, 0.0010, 0.0340, 0.0000, 0.0350, 0.0000, 0.0070,
         0.0000]], device='cuda:0')
